In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
# Configuration for plots
sns.set(style="whitegrid")
plt.rcParams.update({'figure.max_open_warning': 0})

# Define Output Folder
OUTPUT_DIR = 'EDA_Figures'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created output directory: {OUTPUT_DIR}")

# Load the cleaned dataset
file_path = 'clean_final_v3.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: '{file_path}' not found. Please ensure the csv file is in the same directory.")
    exit()

# Ensure numeric columns are actually numeric
numeric_cols = ['Price (VND)', 'RAM (GB)', 'Storage (GB)', 'Screen Size (inch)', 'Refresh Rate (Hz)', 'Weight (kg)']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
# Xem thông tin các cột và kiểu dữ liệu
df.info()

# Kiểm tra giá trị bị thiếu (Missing values)
print("\nSố lượng giá trị thiếu:")
print(df.isnull().sum())

# Chuyển đổi kiểu dữ liệu (nếu cần thiết, ví dụ Price, RAM phải là số)
# Code crawler của bạn đã làm tốt việc này, nhưng kiểm tra lại không thừa.
numeric_cols = ['Price (VND)', 'RAM (GB)', 'Storage (GB)', 'Screen Size (inch)', 'Refresh Rate (Hz)','CPU generation', 'Weight (kg)']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Lọc bỏ các dòng rác (nếu còn sót)
df_clean = df.dropna(subset=['Price (VND)', 'CPU manufacturer'])
print(f"\nDữ liệu sạch sau khi lọc: {len(df_clean)} dòng")
#Save
df_clean.to_csv('clean_final.csv', index=False)
print("Saved merged data to: clean_final.csv")

# Basic Preprocessing for Visualization
df['Manufacturer'] = df['Manufacturer'].str.lower().str.strip()

# Drop 'gpu_class' if it exists to rely on raw 'gpu_model' for analysis
if 'gpu_class' in df.columns:
    df = df.drop(columns=['gpu_class'])

# Helper function to save figures
def save_fig(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, bbox_inches='tight', dpi=300)
    print(f"Saved: {path}")

# ==========================================
# 2. DATA OVERVIEW
# ==========================================
def plot_data_overview():
    summary_data = []
    for col in df.columns:
        non_null = df[col].count()
        total = len(df)
        missing_pct = ((total - non_null) / total) * 100
        dtype = str(df[col].dtype)
        summary_data.append([col, non_null, f"{missing_pct:.1f}%", dtype])

    summary_df = pd.DataFrame(summary_data, columns=['Column Name', 'Non-Null', 'Missing (%)', 'Dtype'])

    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    ax.axis('off')

    table = plt.table(cellText=summary_df.values,
                      colLabels=summary_df.columns,
                      cellLoc='center', loc='center',
                      colColours=['#2c3e50']*4)
    
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.0, 1.8)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(color='white', weight='bold')
            cell.set_facecolor('#2c3e50')
        elif row % 2 == 0:
            cell.set_facecolor('#f8f9fa')

    plt.title('Dataset Structure Overview', fontsize=16, weight='bold', y=0.98)
    save_fig('eda_fig1_overview.png')
    plt.close()

# ==========================================
# 3. UNIVARIATE ANALYSIS
# ==========================================
def plot_price_and_brand():
    # Price Distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Price (VND)'], kde=True, bins=30, color='skyblue')
    plt.title('Distribution of Laptop Prices', fontsize=14)
    plt.xlabel('Price (VND)', fontsize=12)
    save_fig('eda_fig2_price_dist.png')
    plt.close()

    # Manufacturer Counts
    plt.figure(figsize=(10, 6))
    top_brands = df['Manufacturer'].value_counts().nlargest(10).index
    sns.countplot(y='Manufacturer', data=df[df['Manufacturer'].isin(top_brands)], 
                  order=top_brands, palette="viridis")
    plt.title('Top 10 Manufacturers by Model Count', fontsize=14)
    plt.xlabel('Number of Models')
    save_fig('eda_fig3_brand_counts.png')
    plt.close()

# ==========================================
# 4. HARDWARE TRENDS
# ==========================================
def plot_hardware_trends():
    # RAM vs Price
    plt.figure(figsize=(10, 6))
    common_ram = [8.0, 16.0, 32.0, 64.0]
    sns.boxplot(x='RAM (GB)', y='Price (VND)', 
                data=df[df['RAM (GB)'].isin(common_ram)], palette="Blues")
    plt.title('Price Distribution by RAM Capacity', fontsize=14)
    save_fig('eda_fig4_ram_price.png')
    plt.close()

    # GPU Analysis
    top_gpu = df['gpu_model'].value_counts().nlargest(10).index
    df_top_gpu = df[df['gpu_model'].isin(top_gpu)]

    # GPU Counts
    plt.figure(figsize=(12, 8))
    sns.countplot(y='gpu_model', data=df_top_gpu, order=top_gpu, palette="magma")
    plt.title('Top 10 Most Common GPU Models', fontsize=14)
    plt.tight_layout()
    save_fig('eda_fig5_gpu_counts.png')
    plt.close()

    # GPU Prices
    plt.figure(figsize=(14, 8))
    order_price = df_top_gpu.groupby('gpu_model')['Price (VND)'].median().sort_values().index
    sns.boxplot(x='gpu_model', y='Price (VND)', data=df_top_gpu, order=order_price, palette="RdBu")
    plt.title('Price Ranges of Top 10 GPU Models', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_fig('eda_fig6_gpu_price.png')
    plt.close()

# ==========================================
# 5. PORTABILITY & PERFORMANCE
# ==========================================
def plot_portability():
    # Screen vs Weight
    plt.figure(figsize=(11, 7))
    scatter = sns.scatterplot(
        x='Screen Size (inch)', y='Weight (kg)', data=df, 
        hue='Price (VND)', palette='coolwarm', 
        size='Price (VND)', sizes=(30, 150), alpha=0.7
    )
    plt.title('Portability Trade-off: Screen Size vs. Weight', fontsize=14)
    
    # Format Legend
    handles, labels = scatter.get_legend_handles_labels()
    new_labels = [f'{float(lbl)/1e6:.0f}M' if lbl.replace('.','').isdigit() else lbl for lbl in labels]
    plt.legend(handles=handles[1:], labels=new_labels[1:], title='Price (VND)', bbox_to_anchor=(1, 1))
    
    plt.tight_layout()
    save_fig('eda_fig7_screen_weight.png')
    plt.close()

    # Weight vs Price by Top 5 GPUs
    plt.figure(figsize=(10, 6))
    top_5_gpu = df['gpu_model'].value_counts().nlargest(5).index
    df_clean = df[(df['Weight (kg)'] < 4.0) & (df['gpu_model'].isin(top_5_gpu))]
    
    sns.scatterplot(x='Weight (kg)', y='Price (VND)', hue='gpu_model', style='gpu_model',
                    data=df_clean, s=80, alpha=0.7, palette='deep')
    plt.title('Weight vs. Price (Top 5 GPU Models)', fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
    plt.tight_layout()
    save_fig('eda_fig8_gpu_weight.png')
    plt.close()

# ==========================================
# 6. OUTLIER ANALYSIS
# ==========================================
def plot_outliers():
    # Boxplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    sns.boxplot(y=df['Price (VND)'], ax=axes[0], color='skyblue', width=0.5)
    axes[0].set_title('Price Outliers (Valid Premium Segments)', fontsize=12)
    axes[0].ticklabel_format(style='plain', axis='y')
    
    sns.boxplot(y=df['Weight (kg)'], ax=axes[1], color='salmon', width=0.5)
    axes[1].set_title('Weight Outliers (Heavy Performance Units)', fontsize=12)
    
    plt.tight_layout()
    save_fig('eda_fig9_outliers_box.png')
    plt.close()

    # Skewness
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Price (VND)'], kde=True, color='teal', alpha=0.3, bins=30)
    
    mean_val = df['Price (VND)'].mean()
    median_val = df['Price (VND)'].median()
    
    plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val/1e6:.1f}M')
    plt.axvline(median_val, color='blue', linestyle='-', linewidth=2, label=f'Median: {median_val/1e6:.1f}M')
    
    plt.title('Right-Skewed Price Distribution (Post-Cleaning)', fontsize=14)
    plt.legend()
    save_fig('eda_fig10_outliers_skew.png')
    plt.close()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Starting EDA Process...")
    plot_data_overview()
    plot_price_and_brand()
    plot_hardware_trends()
    plot_portability()
    plot_outliers()
    print(f"✅ EDA Complete. All figures saved to folder: '{OUTPUT_DIR}'")

Dataset loaded successfully. Shape: (1246, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1246 entries, 0 to 1245
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Product Name            1246 non-null   object 
 1   Manufacturer            1246 non-null   object 
 2   CPU manufacturer        1246 non-null   object 
 3   CPU brand modifier      1231 non-null   object 
 4   CPU generation          1213 non-null   float64
 5   CPU Speed (GHz)         1124 non-null   float64
 6   RAM (GB)                1246 non-null   float64
 7   RAM Type                1158 non-null   object 
 8   Bus (MHz)               881 non-null    float64
 9   Storage (GB)            1244 non-null   float64
 10  Screen Size (inch)      1246 non-null   float64
 11  Screen Resolution       1234 non-null   object 
 12  Refresh Rate (Hz)       889 non-null    float64
 13  GPU manufacturer        1233 non-null   object

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14768\4288001449.py:120: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(y='Manufacturer', data=df[df['Manufacturer'].isin(top_brands)],


Saved: EDA_Figures\eda_fig3_brand_counts.png


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14768\4288001449.py:134: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='RAM (GB)', y='Price (VND)',


Saved: EDA_Figures\eda_fig4_ram_price.png


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14768\4288001449.py:146: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(y='gpu_model', data=df_top_gpu, order=top_gpu, palette="magma")


Saved: EDA_Figures\eda_fig5_gpu_counts.png


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14768\4288001449.py:155: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='gpu_model', y='Price (VND)', data=df_top_gpu, order=order_price, palette="RdBu")


Saved: EDA_Figures\eda_fig6_gpu_price.png
Saved: EDA_Figures\eda_fig7_screen_weight.png
Saved: EDA_Figures\eda_fig8_gpu_weight.png
Saved: EDA_Figures\eda_fig9_outliers_box.png
Saved: EDA_Figures\eda_fig10_outliers_skew.png
✅ EDA Complete. All figures saved to folder: 'EDA_Figures'
